In [32]:
import random
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict
from langchain_core.messages import BaseMessage

In [33]:
# Carregando as variáveis de ambiente
ENV_PATH = Path('../.env')
load_dotenv(dotenv_path=ENV_PATH)

True

In [34]:
@tool
def throw_dice_tool() -> str:
    """You roll a die and get a person's name."""
    DICE = {
        'ONE': 'Luiz',
        'TWO': 'Two',
        'THREE': 'Nuna',
        'FOUR': 'Henrique',
        'FIVE': 'Egito',
        'SIX': 'Luiz Henrique'
    }
    number = random.choice(list(DICE.keys()))
    return f'O numero do dado foi: {number}, e o nome escolhido foi: {DICE[number]}'

In [35]:
@tool
def current_datetime() -> dict:
    """Returns the current date and time."""
    now = datetime.now()
    return {
        'year': now.year,
        'month': now.month,
        'day': now.day,
        'hour': now.hour,
        'minute': now.minute,
        'second': now.second
    }

TOOLS = [throw_dice_tool, current_datetime]

In [36]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [37]:
# Ele registra as ferramentas no modelo para que o LLM saiba:
# Quais ferramentas ele pode usar
llm_with_tools = llm.bind_tools(TOOLS)

In [38]:
class AgentState(TypedDict):
    messages: List[BaseMessage]

In [39]:
def chatbot(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

In [40]:
tool_node = ToolNode(TOOLS)

In [41]:
def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    # se o modelo pediu tool → vai pra tool
    if last_message.tool_calls:
        return "tools"
    
    # senão → finaliza
    return END

In [42]:
graph = StateGraph(MessagesState)

graph.add_node("chatbot", chatbot)
graph.add_node("tools", tool_node)

graph.set_entry_point("chatbot")

graph.add_conditional_edges(
    "chatbot",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

# depois da tool volta pro LLM
graph.add_edge("tools", "chatbot")

app = graph.compile()

In [43]:
print(app.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +---------+         
          | chatbot |         
          +---------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [44]:
prompt = """
Lance um dado e me diga o número e o nome escolhido.
Além disso me diga a data e hora (ano, mês, dia, hora, minuto e segundo).
"""

result = app.invoke({
    "messages": [HumanMessage(content=prompt)]
})

print(result["messages"][-1].content)

O número do dado foi: **TWO**, e o nome escolhido foi: **Two**.

A data e hora atuais são: **2026-04-17 23:28:44**.
